# Named Entity Recognition for H3 (author_creator and caption structural tags)

### Extract contributor names from author_creator and caption fields and track them across issues.

What we already know about H3 data

Your tokens are already lowercase from 08_text_normalization.ipynb

Fields are short name strings — no need for stopwords or lemmatization

You just need names and which issue they appear in

#### 1. Imports

In [18]:
import pandas as pd
import spacy
import re

nlp = spacy.load("en_core_web_sm")

#### 2. Loading the H3 corpus

In [19]:
df = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/output_files/04_h3_clean.csv")
print(df.shape)
print(df["region_type"].value_counts())
print(df.head())

(1495, 4)
region_type
caption           953
author_creator    542
Name: count, dtype: int64
                                         source_file  page_number  \
0  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
1  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
2  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
3  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
4  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   

      region_type                                               text  
0  author_creator                                               Joan  
1  author_creator                                  Howardena Pindell  
2         caption  Howardena Pindell. Yes — No. Pen and ink on ac...  
3         caption  Gloria Rayl. Untitled. Clay masks pulled from ...  
4  author_creator                                  Marcella Trujillo  


#### 3. Assigning the volume groups

In [20]:
def get_issue_number(filepath):
    match = re.search(r"heresies_(\d+)_combined", filepath)
    return int(match.group(1)) if match else None

def get_volume(issue_nr):
    if issue_nr <= 4:
        return "Vol1_late1970s"
    elif issue_nr <= 8:
        return "Vol2_early1980s"
    elif issue_nr <= 12:
        return "Vol3_late1980s"
    elif issue_nr <= 16:
        return "Vol4_early1990s"
    elif issue_nr <= 20:
        return "Vol5"
    elif issue_nr <= 24:
        return "Vol6"
    else:
        return "Vol7"

df["issue"]  = df["source_file"].apply(get_issue_number)
df["volume"] = df["issue"].apply(get_volume)
df = df[df["issue"] <= 14]

print(df["volume"].value_counts())

volume
Vol1_late1970s     540
Vol3_late1980s     413
Vol2_early1980s    381
Vol4_early1990s    161
Name: count, dtype: int64


#### 4. Extracting the Person names
Using Spacy entity types https://spacy.io/usage/linguistic-features#named-entities

In [21]:
def extract_persons(text):
    if pd.isna(text):
        return []
    doc = nlp(text)
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    return persons

df["persons"] = df["text"].apply(extract_persons)
print("done")
print(df["persons"].head(10))

done
0                                 [Joan]
1                    [Howardena Pindell]
2     [Howardena Pindell, Amy Stromstem]
3                          [Gloria Rayl]
4                    [Marcella Trujillo]
5    [Patron Saint, Guadalupe Tonantzin]
6               [Gloria Jaramillo-Trout]
7                        [Diana Bellesi]
8                                     []
9                                     []
Name: persons, dtype: object


#### 5. Cleaning the names

In [31]:
def clean_persons(persons):
    cleaned = []
    for name in persons:
        # remove possessives
        name = name.replace("'s", "").strip()
        # skip single word names
        if len(name.split()) < 2:
            continue
        # skip very short names
        if len(name) < 5:
            continue
        cleaned.append(name)
    return cleaned

df["persons"] = df["persons"].apply(clean_persons)


# standardize casing
df_contributors["person"] = df_contributors["person"].str.title()

# check result
print(df_contributors["person"].value_counts().head(20))

name_mapping = {
    "Lucy R. Lippard": "Lucy Lippard"
}

df_contributors["person"] = df_contributors["person"].replace(name_mapping)
print(df_contributors["person"].value_counts().head(20))


person
Joan Braderman          12
Su Friedrich            11
Amy Sillman              6
V. E. Browne             5
Dee Shapiro              5
Louise Fishman           5
Roberta Neiman           5
Mary Beth Edelson        5
Howardena Pindell        5
Betsy Damon              4
Valerie Harris           4
Janet Culbertson         4
Lucy R. Lippard          4
Anne Healy               4
Harmony Hammond          4
Joan Jonas               4
Leslie Kanes Weisman     4
Gwendolyn Wright         4
Miriam Schapiro          3
Sarah Mandell            3
Name: count, dtype: int64
person
Joan Braderman          12
Su Friedrich            11
Amy Sillman              6
Dee Shapiro              5
Roberta Neiman           5
Lucy Lippard             5
Louise Fishman           5
V. E. Browne             5
Howardena Pindell        5
Mary Beth Edelson        5
Anne Healy               4
Gwendolyn Wright         4
Leslie Kanes Weisman     4
Joan Jonas               4
Betsy Damon              4
Harmony Hammond

#### 6. Creating one row per Contributor mention

In [33]:
rows = []

for _, row in df.iterrows():
    for person in row["persons"]:
        rows.append({
            "source_file": row["source_file"],
            "issue":       row["issue"],
            "volume":      row["volume"],
            "region_type": row["region_type"],
            "person":      person
        })

df_contributors = pd.DataFrame(rows)
print(df_contributors.shape)
print(df_contributors["person"].value_counts().head(30))

(1113, 5)
person
JOAN BRADERMAN          11
Su Friedrich            10
Amy Sillman              6
Howardena Pindell        5
Mary Beth Edelson        5
Dee Shapiro              5
Roberta Neiman           5
V. E. Browne             5
Louise Fishman           4
Betsy Damon              4
Anne Healy               4
Harmony Hammond          4
Joan Jonas               4
Gwendolyn Wright         4
Valerie Harris           4
Lyn Hughes               3
Cynthia Carr             3
Willi Posey              3
Lynda Benglis            3
Louise Bourgeois         3
Janet Culbertson         3
Leslie Kanes Weisman     3
Susan Saxe               3
Betye Saar               3
Adrienne Rich            3
Merlin Stone             3
Beth Anderson            3
Hannah Wilke             3
Alice Austen             3
Paula Webster            3
Name: count, dtype: int64


#### 7. Save

In [34]:
df_contributors.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/09_h3_contributors.csv", index=False)